# 一、总纲：全章其实只在重复一个"万能配方"

① 数据  →  ② 模型  →  ③ 损失  →  ④ 优化器  →  ⑤ 训练循环

  数据集  →   输出什么？  →   错得多离谱？  →  怎么调参数？  →   迭代N轮


不同的零件把这个配方填了 4 遍
| | 回归（预测数值） | 分类（预测类别） |
|---|---|---|
| **从零实现** | Torch-1 （手写梯度） | Torch-5（手写 softmax） |
| **高级 API** | Torch-2`nn.Linear` + MSE | Torch-6`nn.Linear` + CE |

Torch-1 和 Torch-3 是这两条线的"理论地基"，3.4 是分类需要的"新数据"。

# 二、逐章要点卡

| 节 | 一句话 | 留下的"钩子" |
|---|---|---|
| **Torch-1 线性回归** | 输出是特征的加权和，用 MSE 度量误差 | 权重 W、偏置 b 是模型参数；数据要归一化 |
| **Torch-1 从零实现** | 梯度下降：`w ← w − lr·梯度`，手动算每一步 | 你会亲手调过 W，才懂 API 在干什么 |
| **Torch-2 简洁实现** | 5 行建模型，`nn.Linear`+`SGD` 替掉手写 | 从零 vs API，效果相同 |
| **Torch-3 softmax 回归** | 把 10 个分数压成概率分布，用交叉熵当损失 | **保序**（最大分数=最大概率） |
| **Torch-4 Fashion-MNIST** | 28×28 灰度图，10 类服装，256 一批 | 熟悉数据，后面三章都用它 |
| **Torch-5 从零实现** | 手写 softmax + 交叉熵 + 小批量 sgd | W 是 [784,10]，先 softmax 再算损失 |
| **Torch-6 简洁实现** | softmax 被藏进 `CrossEntropyLoss`，梯度化简成 `p−y` | 输出 logits 而非概率 |

# 三、三条贯穿主线

主线 A：从零实现 → 高级 API，是"先懂原理再图省事"<br>
Torch-1→Torch-2、Torch-5→Torch-6 是同一件事的两个版本。你亲手写过 W -= lr*grad，才看得懂 <br>
trainer.step() 在做什么；亲手写过 param.grad.zero_()，才明白 trainer.zero_grad() 为什么必不可少。<br>
规律：后面第四章（MLP）、第六章（CNN）都沿用"先手写后 API"这个节奏，你已经熟悉了这套流程。<br>

主线 B：回归 vs 分类，本质区别只在"输出层 + 损失"<br>
回归：输出 1 个数 → MSE；<br>
分类：输出 10 个分数 → Softmax 压成概率 → 交叉熵。<br>
其余全部一样：数据集、优化器、训练循环、验证方式。你能把 Torch-2 的代码改成 Torch-6，就是打通了这条线。<br>

主线 C：梯度下降的"方向直觉"贯穿始终<br>
线性回归：MSE 的梯度指向"误差方向"；<br>
分类：交叉熵对 logits 的梯度就是 p − y——负梯度推高真实类别、正梯度压低错误类别。<br>
这个直觉到第四章依然成立：任何模型，本质都是"让真实类别的分数越来越高"。<br>

# 四、核心公式卡

线性回归：   ŷ = X·W + b<br>         损失 = (1/2)(ŷ−y)²<br>
softmax：    p_i = e^{z_i} / Σe^{z_j}     （保序）<br>
交叉熵：     L = −log(p_y)  =  logΣe^{z_j} − z_y<br>
梯度（关键）： ∂L/∂z_i = p_i − y_i<br>
更新：       W ← W − lr·∂L/∂W<br>
准确率：     argmax(p) 是否等于 y<br>

# 五、第三章复盘

1. 为什么 softmax 之后 argmax 和直接对 logits argmax 结果一样？<br>
因为概率和 logites 都是保序的，而 argmax 的作用就是寻找最大值，所以结果都是一样的，因为顺序就没变

2. 手写版和 API 版结果为什么几乎一样？差在哪？<br>
果几乎一样是因为数学完全相同——API 只是把 softmax+交叉熵"合并"了。真正的差别只在数值稳定性：手写版如果分开算 -log(softmax)，当 logit 很大时 e^z 会溢出成 inf → NaN；API 的 log-sum-exp 把指数那一步"和谐"掉了。

3. 为什么分类不用 MSE 用交叉熵？<br>
建模角度：<br>
MSE 的回归对象时具体的浮点数值，来比较彼此之间的大小来看训练的好坏程度，而分类的行为是是对样本进行不同的分类，类别之间的编号，没有比较的意义，不会因为谁离目标编号更近就选谁，那完全没有分类效果，而交叉熵是通过得出 logits 并进行不断地迭代梯度，得到概率最大的那个类<br>
优化角度:<br>
梯度饱和：MSE 套在 softmax 上，输出接近 0/1 时导数被压扁（饱和），训练慢、容易卡住；<br>
交叉熵的梯度是 p − y：误差越大梯度越大，永不饱和，训练又快又稳。

4. trainer.zero_grad() 忘写会发生什么？（梯度会怎样）<br>
梯度下降的更新公式：<br>
$\text{z} \leftarrow \text{z} - \text{lr} \times \text{梯度}$<br>
梯度是负的，− lr × 负数 = 加上一个数 → z 变大；梯度是正的，− lr × 正数 = 减去一个数 → z 变小。负梯度推高真实类别，正梯度压低错误类别

5. 如果预测全对（p_y=1），交叉熵是多少？梯度是多少？这正常吗？<br>
如果全对的话，交叉熵为 0，梯度也是0<br>
数学上它是完美收敛的理想状态；<br>
现实中几乎不可能出现——因为数据本身有噪声，模型不可能每个样本都 p_y=1；<br>
而且如果训练集上真到了 0，往往是过拟合信号（模型把训练集"背"下来了），测试集上 loss 依然远大于 0，acc 可能反而变差。